In [1]:
from pylatexenc.latexwalker import LatexWalker, LatexEnvironmentNode, LatexGroupNode, LatexMacroNode
from pylatexenc.latex2text import LatexNodes2Text

In [2]:
import tarfile
import zipfile
import io
import os
import time
import itertools as itr
import functools as ft
import regex as re
import chardet
from tqdm.auto import tqdm

import pandas as pd
import numpy as np

import gcsfs
fs = gcsfs.GCSFileSystem()

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

# Phase 2
import json
import vertexai
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import storage

from langchain.docstore.document import Document
from langchain_google_vertexai import VertexAI
from langchain.vectorstores import FAISS
#from langchain.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

PROJECT_ID = "arxiv-development"
vertexai.init(project=PROJECT_ID, location="us-central1")

In [3]:
import importlib
import phase_one

In [4]:
from IPython.core.interactiveshell import InteractiveShell
# pretty print all cell's output and not just the last one
InteractiveShell.ast_node_interactivity = "all"

### test phase 1

In [17]:
scopus_df = pd.read_csv("gs://institutional-extract-scratch/training/2311_scopus_17416.csv.zip")
scopus_positive = scopus_df[scopus_df['Primary Org Id'] == 60027550]["ArXiv Id"].unique()
false_positive = [
  "2311.00030v1",
  "2311.00088",
  "2311.00094",
  "2311.00145",
  "2311.00770",
  "2311.02280",
  "2311.02468",
  "2311.03261",
  "2311.03309",
  "2311.03527v2",
  "2311.04856",
  "2311.04862",
  "2311.04942",
  "2311.05674",
  "2311.05678",
  "2311.08133",
  "2311.09521",
  "2311.09562",
  "2311.09638",
  "2311.09734",
  "2311.10140",
  "2311.10604",
  "2311.10985",
  "2311.12103",
  "2311.12152",
  "2311.12263",
  "2311.12334",
  "2311.12656",
  "2311.13107",
  "2311.13244",
  "2311.13718",
  "2311.13900",
  "2311.13907",
  "2311.14103",
  "2311.14409",
  "2311.14599",
  "2311.15441",
  "2311.15533",
  "2311.15541",
  "2311.16400",
  "2311.17162",
  "2311.17252",
  "2311.17314",
  "2311.17844",
  "2311.17915"
]

In [6]:
%%time
res_dict = {}
for arx_id in tqdm(false_positive): #scopus_positive:
    #print(arx_id)
    paper_id = arx_id.split("v")[0]
    res = phase_one.send_one_submission_to_gemini(arx_id)
    #print(f"\n{paper_id}\n{res}")
    res_dict[paper_id] = res


100%|██████████| 45/45 [01:28<00:00,  1.96s/it]

CPU times: user 34.4 s, sys: 492 ms, total: 34.8 s
Wall time: 1min 28s


### Phase 2 Name --> ROR id

Based on FAISS

In [5]:
def load_special_cases_ror():
    docs = []
    ror_sp_gspath = 'gs://institutional-extract-scratch/reference/special_cases.json'
    fs = gcsfs.GCSFileSystem()
    try:
        with fs.open(ror_sp_gspath, "r", encoding="utf-8") as f:
            spec_data = json.load(f)
        for entry in spec:
            ror_id = entry.get("ror_id", "")
            name_loc = entry.get("name_loc", "")
            if name and ror_id:
                content = f"{name_loc} — {ror_id}"
                docs.append(Document(page_content=content))
    except FileNotFoundError:
        docs = []
    return docs


In [6]:
# ROR Prompt
ROR_TEMPLATE = """
You are given an input institution name and a list of ROR entries.

Institution: {question}

Context:
{context}

From the context, pick the best matching ROR ID. If none match, return "null".
DO NOT include explanations, descriptions, or any other text — ONLY the ROR ID or 'null'.
Answer:
"""

In [7]:
# Load the index model, training it if needed.
model_project = 'arxiv-development'
model_bucket_loc = 'institutional-extract-scratch'
dest_blob_name = "models/ror_index.zip"
local_index = "ror_index"


embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

client = storage.Client(project=model_project)
bucket = client.bucket(model_bucket_loc)
blob = bucket.blob(dest_blob_name)
RECREATE_INDEX = False

if RECREATE_INDEX or (not blob.exists()) :
    ror_gspath = 'gs://institutional-extract-scratch/reference/v1.63-2025-04-03-ror-data_schema_v2.json'
    fs = gcsfs.GCSFileSystem()
    with fs.open(ror_gspath, "r", encoding="utf-8") as f:
        ror_data = json.load(f)


    docs = []
    docs = load_special_cases_ror()
    for i,entry in enumerate(ror_data):
        ror_id = entry.get("id", "")
        locs = entry.get('locations')
        loc_name = ""
        try:
            loc_name = f", {locs[0]['geonames_details']['name']}"
        except KeyError:
            pass
        for name_info in entry.get("names", []):
            name = name_info.get("value", "")
            if name and ror_id:
                # format: "name — ROR_id"
                content = f"{name}{loc_name} — {ror_id}"
                docs.append(Document(page_content=content))

    print(f"Prepared {len(docs)} vector entries to build FAISS index")

    # Embedding model (recommended: MiniLM)
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    # Build FAISS index
    vectorstore = FAISS.from_documents(docs, embedding_model)

    # Save index to local file
    vectorstore.save_local(local_index)
    print("ROR vector index built and saved successfully")

    with zipfile.ZipFile(local_index+'.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(local_index):
            for file in files:
                full_path = os.path.join(root, file)
                zipf.write(full_path, os.path.relpath(full_path, local_index))

    client = storage.Client(project=model_project)
    bucket = client.bucket(model_bucket_loc)
    blob = bucket.blob(dest_blob_name)
    blob.upload_from_filename(local_index+'.zip')

    print(f'File uploaded to {dest_blob_name}')

else:
    if not os.path.exists(local_index):
        blob.download_to_filename(local_index+'.zip')
        with zipfile.ZipFile(local_index+'.zip', 'r') as zipf:
            zipf.extractall(local_index)

    vectorstore = FAISS.load_local(
        local_index,
        embeddings=embedding,
        allow_dangerous_deserialization=True
    )




In [8]:
llm = VertexAI(
    model_name="gemini-1.5-flash-002",
    temperature=0,
    max_output_tokens=512,
)

prompt = PromptTemplate(input_variables=["question", "context"], template=ROR_TEMPLATE)

#  Build a Retrieval + QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)


In [9]:
qa_chain.invoke({"query": 'University of California, Los Angeles'})

{'query': 'University of California, Los Angeles',
 'result': '046rm7j60\n',
 'source_documents': [Document(id='3e24bc86-4678-40f8-9c86-957a6103e2ea', metadata={}, page_content='University of California at Los Angeles, Los Angeles — https://ror.org/046rm7j60'),
  Document(id='06d04a98-308f-42e6-8228-78b7c11e02ac', metadata={}, page_content='University of California, Los Angeles, Los Angeles — https://ror.org/046rm7j60')]}

In [10]:
@ft.cache
def get_ror(inst_name):
    try:
        response = qa_chain.invoke({"query": inst_name})
        ror_id = response["result"]

    except Exception as e:
        print(f"Error querying {inst_name}: {e}")
        ror_id = "error"

In [12]:
%%time
get_ror('University of California, Los Angeles')

CPU times: user 5 µs, sys: 0 ns, total: 5 µs
Wall time: 9.3 µs


### Threaded processing for multiple files

## Test

In [ ]:
%%time
results = process_tex_files(false_positive, max_workers=5)

Processing .tex files: 100%|██████████| 45/45 [00:36<00:00,  1.22it/s]

✅ Total processing time: 36.90 seconds
CPU times: user 38.6 s, sys: 1.99 s, total: 40.6 s
Wall time: 36.9 s


In [ ]:
np.array_split(false_positive, len(false_positive)//5)

[array(['2311.00030', '2311.00088', '2311.00094', '2311.00145',
        '2311.00770'], dtype='<U10'),
 array(['2311.02280', '2311.02468', '2311.03261', '2311.03309',
        '2311.03527'], dtype='<U10'),
 array(['2311.04856', '2311.04862', '2311.04942', '2311.05674',
        '2311.05678'], dtype='<U10'),
 array(['2311.08133', '2311.09521', '2311.09562', '2311.09638',
        '2311.09734'], dtype='<U10'),
 array(['2311.10140', '2311.10604', '2311.10985', '2311.12103',
        '2311.12152'], dtype='<U10'),
 array(['2311.12263', '2311.12334', '2311.12656', '2311.13107',
        '2311.13244'], dtype='<U10'),
 array(['2311.13718', '2311.13900', '2311.13907', '2311.14103',
        '2311.14409'], dtype='<U10'),
 array(['2311.14599', '2311.15441', '2311.15533', '2311.15541',
        '2311.16400'], dtype='<U10'),
 array(['2311.17162', '2311.17252', '2311.17314', '2311.17844',
        '2311.17915'], dtype='<U10')]

In [52]:
importlib.reload(phase_one)

<module 'phase_one' from '/home/jupyter/phase_one.py'>

In [ ]:
for key, group in itr.groupby(future.results(), key=lambda x: x[0]):
                inst = {
                    "arxiv_id":key,
                    "institutions_with_ror": [
                        {'name':x[1], 'ror_id':get_ror(x[1])}
                        for x in group
                    ]
                }
                res_list.append(inst)
                #print(future.result())

In [19]:
%%time

import concurrent.futures
import phase_one

def worker(arx_id_list):
    res = phase_one.process_tex_files(arx_id_list)
    return res

def run_in_parallel(arx_id_batches):
    res_list = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(worker, arx_id_list) for arx_id_list in arx_id_batches]
        res = [future.result() for future in concurrent.futures.as_completed(futures)]
    for batch in res:
        for key, group in itr.groupby(res, key=lambda x: x[0]):
            inst = {
                "arxiv_id":key,
                "institutions_with_ror": [
                    {'name':x[1], 'ror_id':get_ror(x[1])}
                    for x in group
                ]
            }
            res_list.append(inst)
            #print(future.result())
    return res_list

os.environ["TOKENIZERS_PARALLELISM"] = "false"    
batches = np.array_split(false_positive, len(false_positive)//5)
res = run_in_parallel(batches)
res

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


✅ Total processing time: 6.59 seconds
✅ Total processing time: 7.33 seconds
✅ Total processing time: 4.79 seconds
✅ Total processing time: 7.94 seconds
✅ Total processing time: 16.05 seconds
✅ Total processing time: 7.87 seconds
✅ Total processing time: 6.96 seconds
✅ Total processing time: 8.16 seconds
✅ Total processing time: 8.91 seconds
Error querying ('2311.04862', '2. University of California, Santa Cruz'): 'tuple' object has no attribute 'replace'
Error querying ('2311.00030', '2. University of California, Berkeley'): 'tuple' object has no attribute 'replace'
Error querying ('2311.09638', '2. Lawrence Berkeley National Laboratory, Berkeley'): 'tuple' object has no attribute 'replace'
Error querying ('2311.10604', '2. Center for Electrochemical Surface Technology, Wiener Neustadt'): 'tuple' object has no attribute 'replace'
Error querying ('2311.03527', '2. University of California, San Diego'): 'tuple' object has no attribute 'replace'
Error querying ('2311.12334', '2. Universit

[{'arxiv_id': ('2311.04862', "1. University of Hawai'i, Hilo"),
  'institutions_with_ror': [{'name': ('2311.04862',
     '2. University of California, Santa Cruz'),
    'ror_id': None}]},
 {'arxiv_id': ('2311.00030', '1. Nazarbayev University, Astana'),
  'institutions_with_ror': [{'name': ('2311.00030',
     '2. University of California, Berkeley'),
    'ror_id': None}]},
 {'arxiv_id': ('2311.09638', '1. University of California, Berkeley'),
  'institutions_with_ror': [{'name': ('2311.09638',
     '2. Lawrence Berkeley National Laboratory, Berkeley'),
    'ror_id': None}]},
 {'arxiv_id': ('2311.10604', '1. TU Wien, Vienna'),
  'institutions_with_ror': [{'name': ('2311.10604',
     '2. Center for Electrochemical Surface Technology, Wiener Neustadt'),
    'ror_id': None}]},
 {'arxiv_id': ('2311.03527', '1. Los Alamos National Laboratory, Los Alamos'),
  'institutions_with_ror': [{'name': ('2311.03527',
     '2. University of California, San Diego'),
    'ror_id': None}]},
 {'arxiv_id': 

In [27]:
phase_one.process_tex_files(['2311.00030v1', '2311.03527v2'])

Processing .tex files:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Total processing time: 1.61 seconds


[('2311.03527', '1. Los Alamos National Laboratory, Los Alamos'),
 ('2311.03527', '2. University of California, San Diego'),
 ('2311.00030', '1. Nazarbayev University, Astana'),
 ('2311.00030', '2. University of California, Berkeley'),
 ('2311.00030', '3. Lawrence Berkeley National Laboratory, Berkeley'),
 ('2311.00030', '4. Korea Astronomy and Space Science Institute, Daejeon'),
 ('2311.00030', '5. University of Science and Technology, Daejeon'),
 ('2311.00030', '6. Technical University of Munich, Garching'),
 ('2311.00030', '7. Max-Planck-Institut fur Astrophysik, Garching')]

In [26]:
%%time 
false_positive = ['2311.00030', '2311.00088', '2311.00094', '2311.00145', '2311.00770', '2311.02280', '2311.02468', '2311.03261', '2311.03309', '2311.03527', '2311.04856', '2311.04862', '2311.04942', '2311.05674', '2311.05678', '2311.08133', '2311.09521', '2311.09562', '2311.09638', '2311.09734', '2311.10140', '2311.10604', '2311.10985', '2311.12103', '2311.12152', '2311.12263', '2311.12334', '2311.12656', '2311.13107', '2311.13244', '2311.13718', '2311.13900', '2311.13907', '2311.14103', '2311.14409', '2311.14599', '2311.15441', '2311.15533', '2311.15541', '2311.16400', '2311.17162', '2311.17252', '2311.17314', '2311.17844', '2311.17915']

def worker(arx_id):
    #import phase_one
    yymm = arx_id.split(".")[0]
    paper_id = arx_id.split("v")[0]
    tar_path = f"gs://arxiv-production-data/ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
    res = phase_one.find_main_tex_source_in_tar(tar_path, encoding='utf-8', all_found=True)
    return res

def run_in_parallel(arx_id_list):
    with concurrent.futures.ProcessPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(worker, arx_id) for arx_id in arx_id_list]
        for future in concurrent.futures.as_completed(futures):
            print(future.result())

batches = np.array_split(false_positive, len(false_positive)//5)
run_in_parallel(false_positive)


KeyboardInterrupt: 

In [ ]:
print(false_positive)

['2311.00030', '2311.00088', '2311.00094', '2311.00145', '2311.00770', '2311.02280', '2311.02468', '2311.03261', '2311.03309', '2311.03527', '2311.04856', '2311.04862', '2311.04942', '2311.05674', '2311.05678', '2311.08133', '2311.09521', '2311.09562', '2311.09638', '2311.09734', '2311.10140', '2311.10604', '2311.10985', '2311.12103', '2311.12152', '2311.12263', '2311.12334', '2311.12656', '2311.13107', '2311.13244', '2311.13718', '2311.13900', '2311.13907', '2311.14103', '2311.14409', '2311.14599', '2311.15441', '2311.15533', '2311.15541', '2311.16400', '2311.17162', '2311.17252', '2311.17314', '2311.17844', '2311.17915']


In [ ]:
import multiprocessing
from multiprocessing import get_context, Pool
import mp_workers

false_positive = ['2311.00030', '2311.00088', '2311.00094', '2311.00145', '2311.00770', '2311.02280', '2311.02468', '2311.03261', '2311.03309', '2311.03527', '2311.04856', '2311.04862', '2311.04942', '2311.05674', '2311.05678', '2311.08133', '2311.09521', '2311.09562', '2311.09638', '2311.09734', '2311.10140', '2311.10604', '2311.10985', '2311.12103', '2311.12152', '2311.12263', '2311.12334', '2311.12656', '2311.13107', '2311.13244', '2311.13718', '2311.13900', '2311.13907', '2311.14103', '2311.14409', '2311.14599', '2311.15441', '2311.15533', '2311.15541', '2311.16400', '2311.17162', '2311.17252', '2311.17314', '2311.17844', '2311.17915']

if __name__ == '__main__':
    with get_context("spawn").Pool(processes=3) as pool:
        results = pool.imap_unordered(mp_workers.worker, false_positive)
        print("Results:", list(results))
  

In [ ]:
import pprint as pp
pp.pprint(results)

In [21]:
importlib.reload(phase_one)

<module 'phase_one' from '/home/jupyter/phase_one.py'>

In [20]:
%%time

import concurrent.futures
import phase_one


def worker(arx_id):
    prd_project = 'arxiv-production'
    prd_bucket_loc = 'arxiv-production-data'    
    #import phase_one
    yymm = arx_id.split(".")[0]
    paper_id = arx_id.split("v")[0]
    tar_path = f"ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
    #return tar_path
    res = phase_one.find_main_tex_source_in_tar(tar_path, encoding='utf-8', all_found=True)
    return res


def run_in_parallel(arx_id_list):
    with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
        futures = [executor.submit(worker, arx_id) for arx_id in arx_id_list]
        #for future in concurrent.futures.as_completed(futures):
        #    print(future.result())
        res = [future for future in concurrent.futures.as_completed(futures)]
        return res
            
res = run_in_parallel(false_positive)
print(len(res))

45
CPU times: user 30.3 ms, sys: 38 ms, total: 68.2 ms
Wall time: 3.37 s


## Scratch

```
# This is formatted as code
```



In [ ]:
[node for node in nodelist if hasattr(node, "macroname") and node.macroname in auth_macros]

In [ ]:
# prompt: use python to read the contents of  gs://arxiv-production-data/ftp/arxiv/papers/2311/2311.15126.abs

#with fs.open('gs://arxiv-production-data/txt/arxiv/2311/2311.15126v3.txt') as f:
with fs.open('gs://arxiv-production-data/txt/arxiv/2311/2311.15533v3.txt') as f:
  content = f.read()
print(content[:200].decode('utf-8'))

In [ ]:
tar_path = "gs://arxiv-production-data/ftp/arxiv/papers/2311/2311.15126.tar.gz"

with fs.open('gs://arxiv-production-data/ftp/arxiv/2311/2311.15533') as f:
  content = f.read()
print(content[:200].decode('utf-8'))

In [ ]:
match_txt = r"""
\author{ Jacob Bedrossian\thanks{\footnotesize Department of Mathematics, University of California, Los Angeles, CA 90095, USA \href{mailto:jacob@math.ucla.edu}{\texttt{jacob@math.ucla.edu}}} \and Siming He\thanks{Department of Mathematics, University of South Carolina, Columbia, SC 29208, USA \href{mailto:siming@mailbox.sc.edu}{\texttt{siming@mailbox.sc.edu}}} \and Sameer Iyer\thanks{Department of Mathematics, University of California, Davis, Davis, CA 95616, USA \href{mailto:sameer@math.ucdavis.edu}{\texttt{sameer@math.ucdavis.edu}}} \and Fei Wang\thanks{School of Mathematical Sciences, CMA-Shanghai, Shanghai Jiao Tong University,
		 Shanghai, China \href{mailto:fwang256@sjtu.edu.cn}{\texttt{fwang256@sjtu.edu.cn}}}}""".strip()

In [ ]:
query_gemini_api(match_txt)

In [ ]:
for extract in src:
  res = query_gemini_api(src)
  if res.startswith("null"):
    continue
  else:
    break

In [ ]:
res

In [ ]:
# prompt: use pandas to load a file from gs bucket

import pandas as pd
df = pd.read_csv('gs://bucket-name/file.csv')

In [ ]:
scopus_df = pd.read_csv("gs://institutional-extract-scratch/training/2311_scopus_17416.csv.zip")

In [ ]:
scopus_df.head()
scopus_df.dtypes

In [ ]:
scopus_df[scopus_df['Primary Org Name'].str.contains("Los Angeles")].head()

In [ ]:
send_one_submission_to_gemini("2311.01623v1")

In [ ]:
scopus_set = set(x.split("v")[0] for x in scopus_positive)
fp_dict = {k:v for k,v in res_dict.items() if k not in scopus_set}
pp.pprint(fp_dict)

In [ ]:
[(k,v) for k,v in res_dict.items() if "null" in v.lower()]

In [ ]:
arx_id = '2311.17314'
print(send_one_submission_to_gemini(arx_id, verbose=False))

In [ ]:
paper_id = "2311.17314" ## download this
yymm = paper_id.split('.')[0]
tar_path = f"gs://arxiv-production-data/ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
candidate_files = find_main_tex_source_in_tar(tar_path, all_found=True, with_weights=True)
print(candidate_files)
src_list = extract_pre_abstract_content(tar_path, candidate_files[1][0])
for src in src_list:
  print(src[:600])

In [ ]:
extract_select_pages_from_txt('gs://arxiv-production-data/txt/arxiv/2311/2311.04473v1.txt')

In [ ]:
txt_path = 'gs://arxiv-production-data/txt/arxiv/2311/2311.04473v1.txt'
with fs.open(txt_path, mode= 'rt', encoding='utf-8') as f:
    file_contents = f.read()

# Split the text by form feed (page break)
contents = file_contents.split("\u000C")

In [ ]:
len(file_contents)
len(contents)
len(contents[0])


In [ ]:
# prompt: pretty print a dictionary

import json

def pretty_print_dict(input_dict):
  """Pretty prints a dictionary to JSON format."""
  print(json.dumps(input_dict, indent=2))

In [ ]:
import pprint as pp
pp.pprint(res_dict)

In [48]:
paper_id = "2311.05674"
yymm = paper_id.split('.')[0]
tar_path = f"gs://arxiv-production-data/ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
candidate_files = find_main_tex_source_in_tar(tar_path, all_found=True)
print(candidate_files)
#src_list = extract_pre_abstract_content(tar_path, candidate_files[0])
#for src in src_list:
#  print(src[:200])

['main.tex']


### Examine extract

In [49]:
paper_id = "2311.05674"
yymm = paper_id.split('.')[0]
tar_path = f"gs://arxiv-production-data/ftp/arxiv/papers/{yymm}/{paper_id}.tar.gz"
tex_main = 'main.tex'

fs = gcsfs.GCSFileSystem()
with fs.open(tar_path, 'rb') as f:
    try:
      with tarfile.open(fileobj=f, mode='r') as in_tar:
        fp = in_tar.extractfile(tex_main)
        wrapped_file = io.TextIOWrapper(fp, newline=None, encoding='utf-8') #universal newlines
        source_text = phase_one.pre_format(wrapped_file.read())
    except UnicodeDecodeError:
        with tarfile.open(fileobj=f, mode='rb') as f:
            raw_data = f.read(10000)
            result = chardet.detect(raw_data)
            detected_encoding = result["encoding"]
        try:
            with tarfile.open(fileobj=f, mode='r') as in_tar:
              fp = in_tar.extractfile(tex_main)
              wrapped_file = io.TextIOWrapper(fp, newline=None, encoding=detected_encoding, errors="replace") #universal newlines
              source_text = phase_one.pre_format(wrapped_file.read())
        except Exception as e:
            tqdm.write(f"Failed to read {tex_file_path} with detected encoding {detected_encoding}: {e}")
            #return ""

# Remove LaTeX comments (lines starting with non-escaped %)
content = re.sub(r"(?<!\\)%.*", "", source_text)
res_list = []

#  "recursive" regex:
#   ((?>[^{}]+|\{(?1)\})*)
# optional brackets
#   (:?\[\d+\])?\s*
# This matches text possibly containing normal characters or nested braces, until the outermost braces are matched.
# If your LaTeX does not have deep nesting, this mainly ensures things like $^{1}$ are correctly parsed.

# Also try parsing latex:
auth_macros = set([
    "author", "auth", "authors",
    "institute", "inst", "institution",
    "affiliation", "affil", "affiliations",
    "address",
    "cmsinstitute",
])
latex_extracted_institutions = []
try:
  lxwkr = LatexWalker(content)
  (nodelist, pos, len_) = lxwkr.get_latex_nodes()
  focus_nodes = [
      (i,node) for i,node in enumerate(nodelist)
      if hasattr(node, "macroname") and node.macroname in auth_macros
  ]
  if focus_nodes:
    for i,node in focus_nodes:
      latex_extracted_institutions.append(node.latex_verbatim())
      try:
        follow_node = nodelist[i+1]
        if isinstance(follow_node, LatexGroupNode):
          latex_extracted_institutions.append(follow_node.latex_verbatim())
      except IndexError:
        pass

  else:
    doc = [
        node for node in nodelist
        if isinstance(node, LatexEnvironmentNode) and node.environmentname=='document'
    ]
    if doc:
      focus_doc_nodes = [
          (i,node) for i, node in enumerate(doc[0].nodelist)
          if isinstance(node, LatexMacroNode) and node.macroname in auth_macros
      ]
      for i, node in focus_doc_nodes:
          latex_extracted_institutions.append(node.latex_verbatim())
          try:
            follow_node = doc[0].nodelist[i+1]
            if isinstance(follow_node, LatexGroupNode):
              latex_extracted_institutions.append(follow_node.latex_verbatim())
          except IndexError:
            pass
  if latex_extracted_institutions:
    res_list.append(latex_extracted_institutions)
except Exception as e:
  print(f"Overly broad except in extract_pre_abstract_content(): {e}")
  pass


institution_patterns = [
    r"\\affiliation\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\institute\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\address\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\inst\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\affil\s*(:?\[\d+\])?\s*\{((?>[^{}]+|\{(?1)\})*)\}",
    r"\\author\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
    r"\\cmsinstitute\s*(:?\[\d+\])?\s*{[^}]+}{([^}]+)}",
]

extracted_institutions = []
for pattern in institution_patterns:
    # Use regex.findall with DOTALL to allow '.' to match newlines
    matches = re.findall(pattern, content, flags=re.DOTALL)
    if matches:
        # Strip each match and add to list
        for m in matches:
            if isinstance(m, tuple):
              extracted_institutions.append(" ".join(m_i for m_i in m))
            else:
              extracted_institutions.extend(m.strip() for m in matches if m.strip())

# If any institution info is extracted, return the deduplicated joined text
if extracted_institutions:
    # You can change the join method; here we join by newline and use set to deduplicate
    #return "\n".join(set(extracted_institutions))
    res_list.append(extracted_institutions)

# If no institution found, try extracting the text before the abstract
match = re.split(
    r"\\begin\s*{\s*abstract\s*}|\\s*\\section\s*{\s*Abstract\s*}",
    content,
    maxsplit=1,
    flags=re.IGNORECASE
)
if len(match) > 1:
    #return match[0].strip()
    res_list.append(match[0].strip())

# If still not found, return the first 1/3 of the content as a fallback
content_length = len(content)
if content_length > 0:
    one_third_length = max(content_length//3, 2000)
    #return content[:one_third_length].strip()
    res_list.append(content[:one_third_length].strip())

len(res_list)
len(focus_nodes)
len(focus_doc_nodes)


3

1

NameError: name 'focus_doc_nodes' is not defined

In [50]:
phase_one.send_one_submission_to_gemini('2311.05674', verbose=True)

Processing ftp/arxiv/papers/2311/2311.05674.tar.gz
0: \author{Maissam Barkeshli${}^1$, Po-Shen Hsin${}^2$, Ryohei Kobayashi${}^1$}

1. Institution Name 1, City
2. Institution Name 2, City



'1. Institution Name 1, City\n2. Institution Name 2, City\n'

In [51]:
res_list

[['\\author{Maissam Barkeshli${}^1$, Po-Shen Hsin${}^2$, Ryohei Kobayashi${}^1$}'],
 '\\documentclass{article}\n\\usepackage[utf8]{inputenc}\n\\usepackage{amsmath,amssymb,amsfonts,stmaryrd}\n\\usepackage{physics}\n\\usepackage{dsfont}\n\\usepackage{graphicx}\n\\usepackage{xcolor}\n\\usepackage{graphicx}\n\\usepackage{cancel}\n\n\\usepackage{hyperref}\n\\usepackage{bm} \n\\usepackage{subfigure} \n\\usepackage{environ}\n\\usepackage{url}\n\\usepackage{hyperref}\n\\usepackage[margin=0.75in]{geometry}\n\n\\newcommand{\\NN}{{\\mathbb N}}\n\\newcommand{\\ZZ}{{\\mathbb Z}}\n\\newcommand{\\Z}{{\\mathbb Z}}\n\\newcommand{\\RR}{{\\mathbb R}}\n\\newcommand{\\CC}{{\\mathbb C}}\n\\newcommand{\\GG}{{\\mathbb G}}\n\\newcommand{\\g}{{\\gamma}}\n\n\n\\newcommand{\\calP}{{\\mathcal{P}}}\n\n\\newcommand{\\om}{\\omega_2}\n\\newcommand{\\ra}{\\rightarrow}\n\\newcommand{\\be}{\\boldsymbol e}\n\\newcommand{\\eps}{\\epsilon}\n\\newcommand{\\br}{{\\mathbf r}}\n\\newcommand{\\da}{{\\delta a}}\n\\newcommand{\\da